In [ ]:
import sqlite3
import random
from faker import Faker
from datetime import datetime, timedelta

# --- Config ---
num_rows = 50000  # total sessions across 3 years
fake = Faker()
db_file_path = "generated_data.db"

session_statuses = ['active', 'paid', 'exited', 'closed', 'overdue']
status_weights = [0.15, 0.05, 0.50, 0.25, 0.05]

entry_station_ids = [1, 2, 3, 4]
exit_station_ids = [5, 6]

def generate_plate():
    """Generates a random Moldovan-style license plate."""
    letters = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    return f"{random.choice(letters)}{random.choice(letters)}{random.choice(letters)}{random.randint(100, 999)}"

def random_date_aug_sep(year: int):
    """Pick a random datetime between 15 Aug – 15 Sep of a given year."""
    start = datetime(year, 8, 15, 0, 0, 0)
    end = datetime(year, 9, 15, 23, 59, 59)
    delta = end - start
    random_sec = random.randint(0, int(delta.total_seconds()))
    return start + timedelta(seconds=random_sec)

sessions = []
current_ticket_id = 10000 

print(f"Generating {num_rows} fake session records...")

for i in range(num_rows):
    ticket_id = None
    exit_time = None
    exit_station = None
    amount_due_cents = 0
    amount_paid_cents = 0
    paid_until = None
    licence_plate_exit = None

    # --- Base Information ---
    status = random.choices(session_statuses, weights=status_weights, k=1)[0]

    # pick year: equally likely 2022, 2023, 2024
    year = random.choice([2021, 2022, 2023, 2024, 2025])
    entry_time = random_date_aug_sep(year)
    licence_plate_entry = generate_plate()
    entry_station = random.choice(entry_station_ids)

    if status in ['exited', 'closed']:
        duration_minutes = random.randint(15, 720)  # 15min – 12h
        duration = timedelta(minutes=duration_minutes)

        exit_time = entry_time + duration
        exit_station = random.choice(exit_station_ids)

        amount_due_cents = int(duration.total_seconds() / 60) * 10
        amount_paid_cents = amount_due_cents

        paid_until = exit_time + timedelta(minutes=15)
        licence_plate_exit = licence_plate_entry if random.random() > 0.05 else generate_plate()

    elif status == 'paid':
        duration_so_far = datetime.now() - entry_time
        amount_due_cents = int(duration_so_far.total_seconds() / 60) * 10
        amount_paid_cents = amount_due_cents
        paid_until = datetime.now() + timedelta(hours=1)

    elif status == 'active':
        duration_so_far = datetime.now() - entry_time
        amount_due_cents = int(duration_so_far.total_seconds() / 60) * 10

    elif status == 'overdue':
        entry_time = random_date_aug_sep(year) - timedelta(days=random.randint(2, 10))
        duration_so_far = datetime.now() - entry_time
        amount_due_cents = int(duration_so_far.total_seconds() / 60) * 10
        amount_paid_cents = 0

    # --- Assign ticket ID ---
    if random.random() < 0.8:
        ticket_id = current_ticket_id
        current_ticket_id += 1

    session_tuple = (
        ticket_id,
        entry_time.isoformat(),
        entry_station,
        exit_time.isoformat() if exit_time else None,
        exit_station,
        status,
        amount_due_cents,
        amount_paid_cents,
        paid_until.isoformat() if paid_until else None,
        licence_plate_entry,
        licence_plate_exit
    )
    sessions.append(session_tuple)

# --- Database Insertion ---
sql = ''' INSERT INTO session(
            ticket_id, entry_time, entry_station, exit_time, exit_station,
            status, amount_due_cents, amount_paid_cents, paid_until,
            licence_plate_entry, licence_plate_exit
          ) VALUES(?,?,?,?,?,?,?,?,?,?,?) '''

try:
    conn = sqlite3.connect(db_file_path)
    cursor = conn.cursor()
    cursor.execute("DELETE FROM session")
    conn.commit()

    
    cursor.executemany(sql, sessions)
    conn.commit()
    
    print(f"✅ Successfully inserted {cursor.rowcount} rows into the 'session' table.")

except sqlite3.Error as error:
    print(f"❌ Failed to insert data into sqlite table: {error}")

finally:
    if conn:
        conn.close()
        print("🔒 The SQLite connection is closed.")


Generating 50000 fake session records...
✅ Successfully inserted 50000 rows into the 'session' table.
🔒 The SQLite connection is closed.
